In [1]:
import tensorflow as tf

In [2]:
ks = tf.keras
kl = ks.layers
cfg = tf.config.experimental

In [3]:
devs = ((None, ), (1000, 1000, 1000, 1000, 1000, 1000), (1000, 1000, 1000, 1000, 1000, 1000))
cfg.set_visible_devices(cfg.get_visible_devices('CPU')[:1], 'CPU')
cfg.set_visible_devices(cfg.get_visible_devices('GPU')[:len(devs) - 1], 'GPU')
for d, ms in zip(cfg.get_visible_devices(), devs):
    vs = [cfg.VirtualDeviceConfiguration(m) for m in ms]
    cfg.set_virtual_device_configuration(d, vs)
devs = cfg.list_logical_devices('CPU')
devs += cfg.list_logical_devices('GPU')
print('devices:', [d.name for d in devs])

devices: ['/job:localhost/replica:0/task:0/device:CPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1', '/job:localhost/replica:0/task:0/device:GPU:2', '/job:localhost/replica:0/task:0/device:GPU:3', '/job:localhost/replica:0/task:0/device:GPU:4', '/job:localhost/replica:0/task:0/device:GPU:5', '/job:localhost/replica:0/task:0/device:GPU:6', '/job:localhost/replica:0/task:0/device:GPU:7', '/job:localhost/replica:0/task:0/device:GPU:8', '/job:localhost/replica:0/task:0/device:GPU:9', '/job:localhost/replica:0/task:0/device:GPU:10', '/job:localhost/replica:0/task:0/device:GPU:11']


In [4]:
tf.config.set_soft_device_placement(False)

In [5]:
class Layer(kl.Layer):
    def __init__(self, i, ps, **kw):
        super().__init__(**kw)
        self.idx = min(i + 1, len(devs) - 1)
        self.ps = ps

    def build(self, input_shape):
        s = input_shape[-1]
        with tf.device(devs[self.idx].name):
            self.w = self.add_weight(name='l_w', shape=(s, s))
            self.b = self.add_weight(name='l_b', shape=(s, ))
        return super().build(input_shape)

    def call(self, x):
        with tf.device(devs[self.idx].name):
            y = tf.einsum('bi,ij->bj', x, self.w) + self.b
        return y

In [6]:
def model_for(ps):
    m = ks.Sequential()
    m.add(kl.Dense(ps.dim_hidden, input_dim=ps.dim_input, name='in'))
    for i in range(ps.num_layers):
        m.add(Layer(i, ps, name=f'lay_{i}'))
    m.add(kl.Dense(ps.dim_input, name='out'))
    m.compile(optimizer=ps.optimizer(), loss=ps.loss(), metrics=[ps.metrics()])
    print(m.summary())
    return m

In [7]:
params = dict(
    dim_hidden=1000,
    dim_input=100,
    loss=ks.losses.MeanAbsoluteError,
    metrics=ks.metrics.MeanAbsoluteError,
    num_layers=10,
    optimizer=ks.optimizers.SGD,
)

In [8]:
class Params:
    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)

In [10]:
ps = Params(**params)
import numpy as np
d = np.ones((100, ps.dim_input))

In [11]:
m = model_for(ps)

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
in (Dense)                   (None, 1000)              101000    
_________________________________________________________________
lay_0 (Layer)                (None, 1000)              1001000   
_________________________________________________________________
lay_1 (Layer)                (None, 1000)              1001000   
_________________________________________________________________
lay_2 (Layer)                (None, 1000)              1001000   
_________________________________________________________________
lay_3 (Layer)                (None, 1000)              1001000   
_________________________________________________________________
lay_4 (Layer)                (None, 1000)              1001000   
_________________________________________________________________
lay_5 (Layer)                (None, 1000)              1

In [12]:
from datetime import datetime
ld = datetime.now().strftime('%Y%m%d-%H%M%S')
ld = f'/tmp/logs/{ld}'
cs = [ks.callbacks.TensorBoard(log_dir=ld, histogram_freq=1)]
m.fit(d, d, callbacks=cs, epochs=10, batch_size=10)

Train on 100 samples
Epoch 1/10
100/100 [==============================] - 2s 17ms/sample - loss: 0.4796 - mean_absolute_error: 0.4796
Epoch 2/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.2872 - mean_absolute_error: 0.2872
Epoch 3/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.2414 - mean_absolute_error: 0.2414
Epoch 4/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.2252 - mean_absolute_error: 0.2252
Epoch 5/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.1988 - mean_absolute_error: 0.1988
Epoch 6/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.1984 - mean_absolute_error: 0.1984
Epoch 7/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.1734 - mean_absolute_error: 0.1734
Epoch 8/10
100/100 [==============================] - 1s 7ms/sample - loss: 0.1551 - mean_absolute_error: 0.1551
Epoch 9/10
100/100 [==============================] - 1s 7ms/sample - loss

In [2]:
%load_ext tensorboard

In [4]:
%tensorboard --logdir /tmp/logs